# UHVDB mined-input counts

Counts sequences / assemblies **mined as toolkit inputs** for UHVDB (not sequences retained after HQ/HC filtering).

1. **Virus sequences** that entered toolkit classify (non-LQ / MQ+ inputs), including r6 DBs — with `source_db` breakdown
2. **Metagenomic assemblies** mined: SPIRE + ENA + LOGAN (**excludes CF_METAG**)
3. **CF bacterial pathogen assemblies** mined from MOTUS_DB

Also documents **hardcoded figure_1b raw virus-DB sizes** (fastas for those original DBs are no longer available).

In [1]:
from pathlib import Path
import gzip

import polars as pl

FIG1 = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_1")
FIG7_UPDATE = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript-update/figure_1")
UHBDB = Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/uhbdb_creation")

# CF pathogen genera — exact genus-name matches only (same as figure_s18.ipynb).
# Does NOT include GTDB letter splits (e.g. Pseudomonas_E, Haemophilus_D).
CF_GENERA = {
    "Pseudomonas",
    "Staphylococcus",
    "Haemophilus",
    "Achromobacter",
    "Stenotrophomonas",
    "Burkholderia",
}

## 0. Hardcoded figure_1b raw virus-DB sizes

From `figure_1b.ipynb` cell **"Human virome database sequence counts"**.
These are **raw downloaded DB sizes** (before non-LQ filtering / toolkit classify).
Original FASTAs are gone; keep these as documentation.

In [2]:
# Documented in figure_1b.ipynb (markdown cell under "Human virome database sequence counts")
FIGURE_1B_RAW_VIRUS_DB_COUNTS = {
    "CHVD": 93860,
    "CNGVC": 93462,
    "CNGVR": 120568,
    "MMGE": 517251,
    "OPD": 189859,
    "OVD": 48425,
    "SMGC": 6935,
    "VMGC": 14224,
    "IMGVR": 708593,  # IMG/VR4 human-associated
}

raw_db_df = pl.DataFrame(
    {"source_db": list(FIGURE_1B_RAW_VIRUS_DB_COUNTS.keys()),
     "n_raw_sequences": list(FIGURE_1B_RAW_VIRUS_DB_COUNTS.values())}
).sort("n_raw_sequences", descending=True)

print("figure_1b hardcoded raw virus-DB total:",
      f"{sum(FIGURE_1B_RAW_VIRUS_DB_COUNTS.values()):,}")
raw_db_df

figure_1b hardcoded raw virus-DB total: 1,793,177


source_db,n_raw_sequences
str,i64
"""IMGVR""",708593
"""MMGE""",517251
"""OPD""",189859
"""CNGVR""",120568
"""CHVD""",93860
"""CNGVC""",93462
"""OVD""",48425
"""VMGC""",14224
"""SMGC""",6935


## 1. Virus sequences that entered toolkit classify

Definition (option B): sequences submitted to UHVDB/toolkit classify as non-LQ / MQ+ inputs.

| Batch | How counted |
|---|---|
| UHGV HQ HC / HQ LC | Input FASTA headers (`uhgv_hq_plus.fna.gz`, `uhgv_hq_lc.fna.gz`) |
| Virus DBs (CHVD…IMGVR) | Unique IDs in `virus_db_non_lq_viruses.txt` |
| Human metagenome viruses (ENA/SPIRE/LOGAN) | Unique IDs in `human_metag_non_lq_viruses.txt` |
| r6 (VIRE, OAVGC, MAGIC, METAVR, NCBIVIRUS) | `results_backup/classify/combined_classify.tsv.gz` rows by `source_db` |

Note: CF_METAG / MOTUS_DB prophage remine IDs are **not** included here (MOTUS assemblies counted in §3; CF_METAG excluded per request).

### 1a. UHGV HQ HC + HQ LC inputs

In [3]:
def count_fasta_headers(path: Path) -> int:
    """Count FASTA headers in plain or gzip-compressed FASTA."""
    opener = gzip.open if str(path).endswith(".gz") else open
    mode = "rt"
    n = 0
    with opener(path, mode) as f:
        for line in f:
            if line.startswith(">"):
                n += 1
    return n

uhgv_hq_hc_fasta = FIG1 / "uhgv_hq_plus.fna.gz"
uhgv_hq_lc_fasta = FIG1 / "uhgv_hq_lc.fna.gz"

uhgv_hq_hc_n = count_fasta_headers(uhgv_hq_hc_fasta)
uhgv_hq_lc_n = count_fasta_headers(uhgv_hq_lc_fasta)

uhgv_counts = pl.DataFrame({
    "source_db": ["UHGV", "UHGV"],
    "subset": ["HQ_HC", "HQ_LC"],
    "n_classify_input": [uhgv_hq_hc_n, uhgv_hq_lc_n],
    "batch": ["uhgv_hq_hc", "uhgv_hq_lc"],
})
print(f"UHGV HQ HC classify input: {uhgv_hq_hc_n:,}")
print(f"UHGV HQ LC classify input: {uhgv_hq_lc_n:,}")
print(f"UHGV total: {uhgv_hq_hc_n + uhgv_hq_lc_n:,}")

# Cross-check vs uhgv_metadata.tsv filters used in figure_1b
uhgv_meta = pl.read_csv(FIG1 / "uhgv_metadata.tsv", separator="\t", ignore_errors=True)
hq = uhgv_meta.filter(pl.col("checkv_completeness") >= 90)
print("uhgv_metadata HQ (completeness>=90):", hq.height)
print("  Uncertain (HQ LC):",
      hq.filter(pl.col("viral_confidence") == "Uncertain").height)
print("  not Uncertain (HQ HC):",
      hq.filter(pl.col("viral_confidence") != "Uncertain").height)
uhgv_counts

UHGV HQ HC classify input: 208,604
UHGV HQ LC classify input: 3,811
UHGV total: 212,415
uhgv_metadata HQ (completeness>=90): 212415
  Uncertain (HQ LC): 3811
  not Uncertain (HQ HC): 208604


source_db,subset,n_classify_input,batch
str,str,i64,str
"""UHGV""","""HQ_HC""",208604,"""uhgv_hq_hc"""
"""UHGV""","""HQ_LC""",3811,"""uhgv_hq_lc"""


### 1b. Virus-DB + human-metagenome non-LQ inputs (figure_1b remine)

These ID lists were the FASTA inputs to the finalized toolkit remine in `figure_1b.ipynb` §5.

In [4]:
# Deduplicate IDs: human_metag list has a small number of duplicate seq_names
virus_db_ids = (
    pl.read_csv(FIG1 / "virus_db_non_lq_viruses.txt", has_header=False, new_columns=["seq_name"])
    .unique("seq_name")
)
human_metag_ids = (
    pl.read_csv(FIG1 / "human_metag_non_lq_viruses.txt", has_header=False, new_columns=["seq_name"])
    .unique("seq_name")
)

# source_db assignments from the same non_lq metadata used to create these lists
non_lq_meta = pl.read_csv(FIG1 / "non_lq_viruses_metadata.csv")

virus_db_by_source = (
    virus_db_ids
    .join(non_lq_meta.select(["seq_name", "source_db"]), on="seq_name", how="left")
    .group_by("source_db")
    .len()
    .rename({"len": "n_classify_input"})
    .with_columns(pl.lit("virus_db_non_lq").alias("batch"))
    .sort("n_classify_input", descending=True)
)

human_metag_by_source = (
    human_metag_ids
    .join(non_lq_meta.select(["seq_name", "source_db"]), on="seq_name", how="left")
    .group_by("source_db")
    .len()
    .rename({"len": "n_classify_input"})
    .with_columns(pl.lit("human_metag_non_lq").alias("batch"))
    .sort("n_classify_input", descending=True)
)

print(f"Virus-DB non-LQ classify input (unique): {virus_db_ids.height:,}")
print(virus_db_by_source)
print(f"\nHuman-metag non-LQ classify input (unique): {human_metag_ids.height:,}")
print(human_metag_by_source)

Virus-DB non-LQ classify input (unique): 715,721
shape: (9, 3)
┌───────────┬──────────────────┬─────────────────┐
│ source_db ┆ n_classify_input ┆ batch           │
│ ---       ┆ ---              ┆ ---             │
│ str       ┆ u32              ┆ str             │
╞═══════════╪══════════════════╪═════════════════╡
│ IMGVR     ┆ 295050           ┆ virus_db_non_lq │
│ OPD       ┆ 95580            ┆ virus_db_non_lq │
│ CNGVC     ┆ 80356            ┆ virus_db_non_lq │
│ MMGE      ┆ 77958            ┆ virus_db_non_lq │
│ CNGVR     ┆ 68431            ┆ virus_db_non_lq │
│ CHVD      ┆ 59477            ┆ virus_db_non_lq │
│ OVD       ┆ 23353            ┆ virus_db_non_lq │
│ VMGC      ┆ 13121            ┆ virus_db_non_lq │
│ SMGC      ┆ 2395             ┆ virus_db_non_lq │
└───────────┴──────────────────┴─────────────────┘

Human-metag non-LQ classify input (unique): 551,468
shape: (3, 3)
┌───────────┬──────────────────┬────────────────────┐
│ source_db ┆ n_classify_input ┆ batch             

### 1c. r6 virus-DB classify inputs (VIRE / OAVGC / MAGIC / METAVR / NCBIVIRUS)

From `uhvdb_v6_creation.ipynb`: `combined_classify.tsv.gz` row count = sequences mined for r6.

In [5]:
v6_classify = pl.read_csv(
    FIG7_UPDATE / "results_backup/classify/combined_classify.tsv.gz",
    separator="\t",
    ignore_errors=True,
    columns=["seq_name", "source_db"],
)

r6_by_source = (
    v6_classify
    .group_by("source_db")
    .len()
    .rename({"len": "n_classify_input"})
    .with_columns(pl.lit("r6").alias("batch"))
    .sort("n_classify_input", descending=True)
)

print(f"r6 classify input total: {v6_classify.height:,}")
r6_by_source

r6 classify input total: 1,261,909


source_db,n_classify_input,batch
str,u32,str
"""VIRE""",934217,"""r6"""
"""OAVGC""",128829,"""r6"""
"""MAGIC""",90157,"""r6"""
"""NCBIVIRUS""",77277,"""r6"""
"""METAVR""",31429,"""r6"""


### 1d. Combined virus classify-input summary

In [6]:
virus_classify_inputs = pl.concat(
    [
        uhgv_counts.select(["source_db", "n_classify_input", "batch"]),
        virus_db_by_source.select(["source_db", "n_classify_input", "batch"]),
        human_metag_by_source.select(["source_db", "n_classify_input", "batch"]),
        r6_by_source.select(["source_db", "n_classify_input", "batch"]),
    ],
    how="vertical_relaxed",
)

virus_by_source = (
    virus_classify_inputs
    .group_by("source_db")
    .agg(pl.col("n_classify_input").sum())
    .sort("n_classify_input", descending=True)
)

virus_by_batch = (
    virus_classify_inputs
    .group_by("batch")
    .agg(pl.col("n_classify_input").sum())
    .sort("n_classify_input", descending=True)
)

total_virus_classify_input = virus_classify_inputs["n_classify_input"].sum()
source_db_list = virus_by_source["source_db"].to_list()

print("=== Virus sequences that entered toolkit classify ===")
print(f"TOTAL: {total_virus_classify_input:,}")
print(f"source_dbs ({len(source_db_list)}): {source_db_list}")
print("\nBy source_db:")
print(virus_by_source)
print("\nBy mining batch:")
print(virus_by_batch)

# Optional: compare to figure_1b mining-summary viral (post non-viral filter) counts
FIGURE_1B_VIRAL_AFTER_CLASSIFY = {
    "UHGV HQ HC": 207700,
    "UHGV HQ LC": 3630,
    "VirusDB": 715410,
    "Human Metag": 547242,
}
print("\n[reference] figure_1b mining summary (viral after non-viral filter, not input):",
      f"{sum(FIGURE_1B_VIRAL_AFTER_CLASSIFY.values()):,}")

=== Virus sequences that entered toolkit classify ===
TOTAL: 2,741,754
source_dbs (18): ['VIRE', 'SPIRE', 'IMGVR', 'UHGV', 'ENA', 'OAVGC', 'OPD', 'MAGIC', 'CNGVC', 'MMGE', 'NCBIVIRUS', 'LOGAN', 'CNGVR', 'CHVD', 'METAVR', 'OVD', 'VMGC', 'SMGC']

By source_db:
shape: (18, 2)
┌───────────┬──────────────────┐
│ source_db ┆ n_classify_input │
│ ---       ┆ ---              │
│ str       ┆ i64              │
╞═══════════╪══════════════════╡
│ VIRE      ┆ 934217           │
│ SPIRE     ┆ 331177           │
│ IMGVR     ┆ 295050           │
│ UHGV      ┆ 212415           │
│ ENA       ┆ 144342           │
│ …         ┆ …                │
│ CHVD      ┆ 59477            │
│ METAVR    ┆ 31429            │
│ OVD       ┆ 23353            │
│ VMGC      ┆ 13121            │
│ SMGC      ┆ 2395             │
└───────────┴──────────────────┘

By mining batch:
shape: (5, 2)
┌────────────────────┬──────────────────┐
│ batch              ┆ n_classify_input │
│ ---                ┆ ---              │
│ str  

## 2. Metagenomic assemblies mined (SPIRE + ENA + LOGAN)

Counts rows in the human airways/skin/urogenital samplesheets created in `figure_1b.ipynb` §4.

**CF_METAG is excluded** (per request).

In [7]:
assembly_sheets = {
    "ENA": FIG1 / "ena_human_samplesheet.csv.gz",
    "LOGAN": FIG1 / "logan_human_samplesheet.csv.gz",
    "SPIRE": FIG1 / "spire_human_samplesheet.csv.gz",
}

assembly_rows = []
for source_db, path in assembly_sheets.items():
    df = pl.read_csv(path)
    assembly_rows.append({"source_db": source_db, "n_assemblies": df.height})

assembly_counts = pl.DataFrame(assembly_rows).sort("n_assemblies", descending=True)
total_assemblies = assembly_counts["n_assemblies"].sum()

print("=== Metagenomic assemblies mined (excl. CF_METAG) ===")
print(f"TOTAL: {total_assemblies:,}")
print("(figure_1b markdown claimed Total assemblies = 79,131)")
assembly_counts

=== Metagenomic assemblies mined (excl. CF_METAG) ===
TOTAL: 79,131
(figure_1b markdown claimed Total assemblies = 79,131)


source_db,n_assemblies
str,i64
"""LOGAN""",54908
"""ENA""",14603
"""SPIRE""",9620


## 3. CF bacterial pathogen assemblies mined (MOTUS_DB)

Filter MOTUS MAG + isolate samplesheets to **exact** CF pathogen genus names
(same filter as `figure_s18.ipynb`: `genus == 'Pseudomonas'`, etc.):
`Pseudomonas`, `Staphylococcus`, `Haemophilus`, `Achromobacter`, `Stenotrophomonas`, `Burkholderia`.

GTDB letter splits (`Pseudomonas_E`, `Haemophilus_D`, …) are **not** included.

In [8]:
motus_sheets = [
    (UHBDB / "motusdb_mag_samplesheet.tsv", "MAG"),
    (UHBDB / "motusdb_isolates_samplesheet.tsv", "Isolate"),
]

motus_parts = []
for path, source_type in motus_sheets:
    df = (
        pl.read_csv(path, separator="\t")
        .filter(pl.col("genus").is_in(sorted(CF_GENERA)))
        .with_columns(pl.lit(source_type).alias("source_type"))
    )
    motus_parts.append(df)

motus_cf = pl.concat(motus_parts, how="vertical_relaxed")

motus_by_genus = (
    motus_cf
    .group_by(["genus", "source_type"])
    .len()
    .rename({"len": "n_assemblies"})
    .sort(["genus", "source_type"])
)

motus_by_genus_total = (
    motus_cf
    .group_by("genus")
    .len()
    .rename({"len": "n_assemblies"})
    .sort("n_assemblies", descending=True)
)

motus_by_type = (
    motus_cf
    .group_by("source_type")
    .len()
    .rename({"len": "n_assemblies"})
    .sort("n_assemblies", descending=True)
)

print("=== CF pathogen MOTUS_DB assemblies mined (exact genus) ===")
print(f"TOTAL: {motus_cf.height:,}")
print("\nBy genus:")
print(motus_by_genus_total)
print("\nBy MAG vs Isolate:")
print(motus_by_type)
print("\nBy genus × source_type:")
print(motus_by_genus)

=== CF pathogen MOTUS_DB assemblies mined (exact genus) ===
TOTAL: 52,592

By genus:
shape: (6, 2)
┌──────────────────┬──────────────┐
│ genus            ┆ n_assemblies │
│ ---              ┆ ---          │
│ str              ┆ u32          │
╞══════════════════╪══════════════╡
│ Staphylococcus   ┆ 37734        │
│ Pseudomonas      ┆ 7684         │
│ Burkholderia     ┆ 4559         │
│ Haemophilus      ┆ 1311         │
│ Stenotrophomonas ┆ 985          │
│ Achromobacter    ┆ 319          │
└──────────────────┴──────────────┘

By MAG vs Isolate:
shape: (2, 2)
┌─────────────┬──────────────┐
│ source_type ┆ n_assemblies │
│ ---         ┆ ---          │
│ str         ┆ u32          │
╞═════════════╪══════════════╡
│ Isolate     ┆ 47381        │
│ MAG         ┆ 5211         │
└─────────────┴──────────────┘

By genus × source_type:
shape: (12, 3)
┌──────────────────┬─────────────┬──────────────┐
│ genus            ┆ source_type ┆ n_assemblies │
│ ---              ┆ ---         ┆ ---         

## 4. Grand summary

In [9]:
summary = pl.DataFrame({
    "metric": [
        "Virus sequences entered toolkit classify (incl. r6)",
        "Metagenomic assemblies mined (SPIRE+ENA+LOGAN)",
        "CF pathogen MOTUS_DB assemblies mined",
        "[doc] figure_1b raw virus-DB sizes (pre-filter, hardcoded)",
    ],
    "count": [
        total_virus_classify_input,
        total_assemblies,
        motus_cf.height,
        sum(FIGURE_1B_RAW_VIRUS_DB_COUNTS.values()),
    ],
})

print(summary)
print("\nVirus source_dbs:", source_db_list)
print("Assembly source_dbs:", assembly_counts["source_db"].to_list())
print("MOTUS CF genera:", sorted(CF_GENERA))

shape: (4, 2)
┌─────────────────────────────────┬─────────┐
│ metric                          ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ i64     │
╞═════════════════════════════════╪═════════╡
│ Virus sequences entered toolki… ┆ 2741754 │
│ Metagenomic assemblies mined (… ┆ 79131   │
│ CF pathogen MOTUS_DB assemblie… ┆ 52592   │
│ [doc] figure_1b raw virus-DB s… ┆ 1793177 │
└─────────────────────────────────┴─────────┘

Virus source_dbs: ['VIRE', 'SPIRE', 'IMGVR', 'UHGV', 'ENA', 'OAVGC', 'OPD', 'MAGIC', 'CNGVC', 'MMGE', 'NCBIVIRUS', 'LOGAN', 'CNGVR', 'CHVD', 'METAVR', 'OVD', 'VMGC', 'SMGC']
Assembly source_dbs: ['LOGAN', 'ENA', 'SPIRE']
MOTUS CF genera: ['Achromobacter', 'Burkholderia', 'Haemophilus', 'Pseudomonas', 'Staphylococcus', 'Stenotrophomonas']
